# PyVSC Tutorial

PyVSC는 SystemVerilog의 **Constrained Random** + **Functional Coverage**를 Python으로 구현한 라이브러리입니다.

**핵심 기능:**
- `@vsc.randobj`: 랜덤 변수를 가진 클래스 정의 (SV의 `rand` 변수)
- `@vsc.constraint`: 제약조건 정의 (SV의 `constraint`)
- `@vsc.covergroup`: Coverage 그룹 정의
- `vsc.coverpoint`: Coverpoint 정의
- `vsc.cross`: Cross coverage 정의

In [ ]:
import vsc
import numpy as np

---
## 1. 기본 Constrained Random

SystemVerilog와 비교:
```systemverilog
class my_item;
    rand bit [7:0] a;
    rand bit [7:0] b;
    
    constraint ab_c {
        a < b;
    }
endclass
```

In [ ]:
# PyVSC 버전
@vsc.randobj
class my_item:
    def __init__(self):
        self.a = vsc.rand_bit_t(8)  # rand bit [7:0] a
        self.b = vsc.rand_bit_t(8)  # rand bit [7:0] b
    
    @vsc.constraint
    def ab_c(self):
        self.a < self.b  # constraint: a < b

# 인스턴스 생성 및 randomize
item = my_item()

print("Randomize 10 times (constraint: a < b):")
for i in range(10):
    item.randomize()
    print(f"  [{i}] a={item.a:3d}, b={item.b:3d}, a<b? {item.a < item.b}")

---
## 2. 다양한 데이터 타입

In [ ]:
@vsc.randobj
class data_types_demo:
    def __init__(self):
        # Unsigned types
        self.u8 = vsc.rand_uint8_t()      # 0 ~ 255
        self.u16 = vsc.rand_uint16_t()    # 0 ~ 65535
        self.u32 = vsc.rand_uint32_t()    # 0 ~ 2^32-1
        
        # Signed types
        self.s8 = vsc.rand_int8_t()       # -128 ~ 127
        self.s16 = vsc.rand_int16_t()     # -32768 ~ 32767
        
        # Bit type with specific width
        self.bits_4 = vsc.rand_bit_t(4)   # 0 ~ 15
        self.bits_12 = vsc.rand_bit_t(12) # 0 ~ 4095

demo = data_types_demo()
demo.randomize()

print("Data types demo:")
print(f"  u8:  {demo.u8} (0~255)")
print(f"  u16: {demo.u16} (0~65535)")
print(f"  s8:  {demo.s8} (-128~127)")
print(f"  s16: {demo.s16} (-32768~32767)")
print(f"  bits_4:  {demo.bits_4} (0~15)")
print(f"  bits_12: {demo.bits_12} (0~4095)")

---
## 3. 다양한 Constraint 표현

SystemVerilog의 constraint 문법을 Python으로 표현하는 방법들

In [ ]:
@vsc.randobj
class constraint_demo:
    def __init__(self):
        self.a = vsc.rand_uint8_t()
        self.b = vsc.rand_uint8_t()
        self.c = vsc.rand_uint8_t()
    
    @vsc.constraint
    def range_c(self):
        # 범위 제약: 10 <= a <= 50
        self.a >= 10
        self.a <= 50
    
    @vsc.constraint
    def inside_c(self):
        # inside 제약: b는 특정 값들 중 하나
        self.b.inside(vsc.rangelist(1, 2, 4, 8, 16, 32, 64, 128))
    
    @vsc.constraint
    def relation_c(self):
        # 관계 제약: c = a + b
        self.c == self.a + self.b

demo = constraint_demo()
print("Constraint demo (10<=a<=50, b in {1,2,4,8,16,32,64,128}, c=a+b):")
for i in range(5):
    demo.randomize()
    print(f"  a={demo.a:3d}, b={demo.b:3d}, c={demo.c:3d}, c==a+b? {demo.c == demo.a + demo.b}")

In [ ]:
# if-else constraint
@vsc.randobj
class if_else_constraint:
    def __init__(self):
        self.mode = vsc.rand_bit_t(2)  # 0, 1, 2, 3
        self.value = vsc.rand_uint8_t()
    
    @vsc.constraint
    def mode_value_c(self):
        # mode에 따라 value 범위가 다름
        with vsc.if_then(self.mode == 0):
            self.value.inside(vsc.rangelist(vsc.rng(0, 10)))
        with vsc.else_if(self.mode == 1):
            self.value.inside(vsc.rangelist(vsc.rng(11, 50)))
        with vsc.else_if(self.mode == 2):
            self.value.inside(vsc.rangelist(vsc.rng(51, 100)))
        with vsc.else_then:
            self.value.inside(vsc.rangelist(vsc.rng(101, 200)))

demo = if_else_constraint()
print("If-else constraint (mode determines value range):")
for i in range(8):
    demo.randomize()
    print(f"  mode={demo.mode}, value={demo.value:3d}")

---
## 4. Soft Constraints

Soft constraint는 가능하면 만족시키지만, hard constraint와 충돌하면 무시됨

In [ ]:
@vsc.randobj
class soft_constraint_demo:
    def __init__(self):
        self.a = vsc.rand_uint8_t()
        self.b = vsc.rand_uint8_t()
    
    @vsc.constraint
    def hard_c(self):
        # Hard constraint: 항상 만족
        self.a <= 100
        self.b <= 100
    
    @vsc.constraint
    def soft_c(self):
        # Soft constraint: 가능하면 a=50
        vsc.soft(self.a == 50)

demo = soft_constraint_demo()
print("Soft constraint (prefer a=50):")
for i in range(5):
    demo.randomize()
    print(f"  a={demo.a:3d}, b={demo.b:3d}")

In [ ]:
# randomize_with로 inline constraint 추가 (soft 무시됨)
print("\nWith inline constraint (a < 30, soft ignored):")
for i in range(5):
    with demo.randomize_with() as it:
        it.a < 49  # inline hard constraint
    print(f"  a={demo.a:3d}, b={demo.b:3d}")

---
## 5. 기본 Covergroup 정의

SystemVerilog와 비교:
```systemverilog
covergroup my_cg with function sample(bit [3:0] a);
    cp_a: coverpoint a {
        bins low = {[0:3]};
        bins mid = {[4:11]};
        bins high = {[12:15]};
    }
endgroup
```

In [ ]:
# PyVSC 버전
@vsc.covergroup
class my_covergroup:
    def __init__(self):
        # sample 함수의 파라미터 정의
        self.with_sample(a=vsc.bit_t(4))
        
        # Coverpoint 정의
        self.cp_a = vsc.coverpoint(self.a, bins={
            "low": vsc.bin_array([], [0, 3]),    # 0~3
            "mid": vsc.bin_array([], [4, 11]),   # 4~11
            "high": vsc.bin_array([], [12, 15])  # 12~15
        })

# Covergroup 인스턴스 생성
cg = my_covergroup()

# 샘플링
print("Sampling values:")
for val in [0, 5, 12, 3, 8, 15, 1, 10]:
    cg.sample(val)
    print(f"  Sampled {val:2d}, coverage: {cg.get_inst_coverage():.1f}%")

print(f"\nFinal coverage: {cg.get_inst_coverage():.1f}%")

In [ ]:
# Coverage 리포트 출력
vsc.report_coverage(details=True)

---
## 6. Cross Coverage

두 coverpoint의 모든 조합을 tracking

In [ ]:
@vsc.covergroup
class cross_coverage_demo:
    def __init__(self):
        self.with_sample(
            x=vsc.int8_t(),
            y=vsc.int8_t()
        )
        
        # x의 부호
        self.cp_x_sign = vsc.coverpoint(self.x, bins={
            "negative": vsc.bin([-128, -1]),
            "zero": vsc.bin(0),
            "positive": vsc.bin([1, 127])
        })
        
        # y의 부호
        self.cp_y_sign = vsc.coverpoint(self.y, bins={
            "negative": vsc.bin([-128, -1]),
            "zero": vsc.bin(0),
            "positive": vsc.bin([1, 127])
        })
        
        # Cross: x_sign x y_sign (9가지 조합)
        self.cross_xy = vsc.cross([self.cp_x_sign, self.cp_y_sign])

cg = cross_coverage_demo()

# 다양한 조합 샘플링
test_pairs = [
    (-10, -5),   # neg x neg
    (-10, 0),    # neg x zero
    (-10, 5),    # neg x pos
    (0, -5),     # zero x neg
    (0, 0),      # zero x zero
    (0, 5),      # zero x pos
    (10, -5),    # pos x neg
    (10, 0),     # pos x zero
    (10, 5),     # pos x pos
]

print("Cross coverage sampling (x_sign x y_sign):")
for x, y in test_pairs:
    cg.sample(x, y)
    print(f"  ({x:4d}, {y:4d}) -> coverage: {cg.get_inst_coverage():.1f}%")

print(f"\nFinal cross coverage: {cg.get_inst_coverage():.1f}%")

In [ ]:
vsc.report_coverage(details=True)

---
## 7. 실전 예제: Inner Product Coverage

벡터의 inner product에서 다양한 케이스를 커버하는 예제

In [ ]:
# Constrained random vector element
@vsc.randobj
class vector_element:
    def __init__(self):
        self.value = vsc.rand_int16_t()
    
    @vsc.constraint
    def range_c(self):
        self.value >= -1000
        self.value <= 1000


# Coverage for inner product pairs
@vsc.covergroup
class inner_product_coverage:
    def __init__(self):
        self.with_sample(
            x=vsc.int16_t(),  # input element
            w=vsc.int16_t()   # weight element
        )
        
        # x의 크기 분류
        self.cp_x_magnitude = vsc.coverpoint(self.x, bins={
            "large_neg": vsc.bin([-1000, -100]),
            "small_neg": vsc.bin([-99, -1]),
            "zero": vsc.bin(0),
            "small_pos": vsc.bin([1, 99]),
            "large_pos": vsc.bin([100, 1000])
        })
        
        # w의 크기 분류
        self.cp_w_magnitude = vsc.coverpoint(self.w, bins={
            "large_neg": vsc.bin([-1000, -100]),
            "small_neg": vsc.bin([-99, -1]),
            "zero": vsc.bin(0),
            "small_pos": vsc.bin([1, 99]),
            "large_pos": vsc.bin([100, 1000])
        })
        
        # Cross coverage: 25가지 조합 (5 x 5)
        self.cross_xw = vsc.cross([self.cp_x_magnitude, self.cp_w_magnitude])


# 테스트 실행
x_elem = vector_element()
w_elem = vector_element()
cov = inner_product_coverage()

print("Inner product coverage test:")
print("Generating random (x, w) pairs and tracking coverage...\n")

for i in range(100):
    x_elem.randomize()
    w_elem.randomize()
    
    cov.sample(x_elem.value, w_elem.value)
    
    if (i + 1) % 20 == 0:
        print(f"  After {i+1} samples: {cov.get_inst_coverage():.1f}% coverage")

print(f"\nFinal coverage: {cov.get_inst_coverage():.1f}%")

In [ ]:
# 상세 리포트
vsc.report_coverage(details=True)

---
## 8. Coverage-Driven Random Generation

Coverage가 낮은 bin을 우선적으로 채우는 방법

In [ ]:
# Coverage 초기화
@vsc.covergroup
class targeted_coverage:
    def __init__(self):
        self.with_sample(val=vsc.uint8_t())
        
        self.cp_val = vsc.coverpoint(self.val, bins={
            "bin_0": vsc.bin([0, 49]),
            "bin_1": vsc.bin([50, 99]),
            "bin_2": vsc.bin([100, 149]),
            "bin_3": vsc.bin([150, 199]),
            "bin_4": vsc.bin([200, 255])
        })


@vsc.randobj
class targeted_random:
    def __init__(self):
        self.val = vsc.rand_uint8_t()
        self.target_bin = 0  # 목표 bin
    
    @vsc.constraint
    def target_c(self):
        # target_bin에 따라 다른 범위 제약
        with vsc.if_then(self.target_bin == 0):
            self.val.inside(vsc.rangelist(vsc.rng(0, 49)))
        with vsc.else_if(self.target_bin == 1):
            self.val.inside(vsc.rangelist(vsc.rng(50, 99)))
        with vsc.else_if(self.target_bin == 2):
            self.val.inside(vsc.rangelist(vsc.rng(100, 149)))
        with vsc.else_if(self.target_bin == 3):
            self.val.inside(vsc.rangelist(vsc.rng(150, 199)))
        with vsc.else_then:
            self.val.inside(vsc.rangelist(vsc.rng(200, 255)))


cov = targeted_coverage()
rand_obj = targeted_random()

print("Coverage-driven generation:")
print("Targeting each bin sequentially...\n")

# 각 bin을 순차적으로 타겟팅
for target in range(5):
    rand_obj.target_bin = target
    rand_obj.randomize()
    cov.sample(rand_obj.val)
    print(f"  Target bin {target}: generated {rand_obj.val:3d}, coverage: {cov.get_inst_coverage():.1f}%")

print(f"\nFinal coverage: {cov.get_inst_coverage():.1f}%")

---
## 9. Bin Array와 Auto Bins

In [ ]:
@vsc.covergroup
class bin_array_demo:
    def __init__(self):
        self.with_sample(a=vsc.uint8_t())
        
        # bin_array: 자동으로 여러 bin 생성
        # bin_array([N], range) -> N개의 bin으로 range를 나눔
        self.cp_a = vsc.coverpoint(self.a, bins={
            "quarters": vsc.bin_array([4], [0, 255])  # 4개 bin으로 0-255 분할
        })

cg = bin_array_demo()

# 샘플링
test_values = [10, 70, 130, 200, 50, 100, 150, 250]
for v in test_values:
    cg.sample(v)

print("Bin array demo (4 bins for 0-255):")
vsc.report_coverage(details=True)

---
## 10. 실전 예제: FPINT GEMM Coverage

행렬 곱셈에서 다양한 입력 조합을 커버

In [ ]:
# GEMM 테스트 벡터
@vsc.randobj
class gemm_test_vector:
    def __init__(self):
        # Matrix dimensions
        self.M = vsc.rand_uint16_t()
        self.K = vsc.rand_uint16_t()
        self.N = vsc.rand_uint16_t()
        
        # Value characteristics
        self.input_range = vsc.rand_uint8_t()   # 0=small, 1=medium, 2=large
        self.weight_range = vsc.rand_uint8_t()  # 0=small, 1=medium, 2=large
    
    @vsc.constraint
    def dim_c(self):
        # 행렬 크기 제약
        self.M >= 1
        self.M <= 64
        self.K.inside(vsc.rangelist(16, 32, 64, 128))  # K는 특정 값만
        self.N >= 1
        self.N <= 64
    
    @vsc.constraint
    def range_c(self):
        self.input_range <= 2
        self.weight_range <= 2


# GEMM Coverage
@vsc.covergroup
class gemm_coverage:
    def __init__(self):
        self.with_sample(
            M=vsc.uint16_t(),
            K=vsc.uint16_t(),
            N=vsc.uint16_t(),
            input_range=vsc.uint8_t(),
            weight_range=vsc.uint8_t()
        )
        
        # Matrix size bins
        self.cp_M = vsc.coverpoint(self.M, bins={
            "tiny": vsc.bin([1, 4]),
            "small": vsc.bin([5, 16]),
            "medium": vsc.bin([17, 32]),
            "large": vsc.bin([33, 64])
        })
        
        self.cp_K = vsc.coverpoint(self.K, bins={
            "k16": vsc.bin(16),
            "k32": vsc.bin(32),
            "k64": vsc.bin(64),
            "k128": vsc.bin(128)
        })
        
        self.cp_N = vsc.coverpoint(self.N, bins={
            "tiny": vsc.bin([1, 4]),
            "small": vsc.bin([5, 16]),
            "medium": vsc.bin([17, 32]),
            "large": vsc.bin([33, 64])
        })
        
        # Value range bins
        self.cp_input = vsc.coverpoint(self.input_range, bins={
            "small": vsc.bin(0),
            "medium": vsc.bin(1),
            "large": vsc.bin(2)
        })
        
        self.cp_weight = vsc.coverpoint(self.weight_range, bins={
            "small": vsc.bin(0),
            "medium": vsc.bin(1),
            "large": vsc.bin(2)
        })
        
        # Cross coverage
        self.cross_MN = vsc.cross([self.cp_M, self.cp_N])  # 16 combinations
        self.cross_input_weight = vsc.cross([self.cp_input, self.cp_weight])  # 9 combinations


# 테스트 실행
tv = gemm_test_vector()
cov = gemm_coverage()

print("GEMM Coverage Test")
print("="*50)

for i in range(50):
    tv.randomize()
    cov.sample(tv.M, tv.K, tv.N, tv.input_range, tv.weight_range)
    
    if (i + 1) % 10 == 0:
        print(f"After {i+1} tests: {cov.get_inst_coverage():.1f}% coverage")

print(f"\nFinal coverage: {cov.get_inst_coverage():.1f}%")

In [ ]:
# 상세 리포트
vsc.report_coverage(details=True)

---
## 11. Coverage 저장/로드

In [ ]:
# Coverage를 XML 파일로 저장
vsc.write_coverage_db('gemm_coverage.xml')
print("Coverage saved to gemm_coverage.xml")

# 나중에 로드하려면:
# vsc.load_coverage_db('gemm_coverage.xml')

---
## 12. 요약: SystemVerilog vs PyVSC

| SystemVerilog | PyVSC |
|--------------|-------|
| `rand bit [7:0] a` | `self.a = vsc.rand_bit_t(8)` |
| `constraint c { a < b; }` | `@vsc.constraint` + `self.a < self.b` |
| `a inside {1, 2, 3}` | `self.a.inside(vsc.rangelist(1, 2, 3))` |
| `soft constraint` | `vsc.soft(...)` |
| `covergroup` | `@vsc.covergroup` |
| `coverpoint a { bins... }` | `vsc.coverpoint(self.a, bins={...})` |
| `cross cp1, cp2` | `vsc.cross([self.cp1, self.cp2])` |
| `bins b[4] = {[0:255]}` | `vsc.bin_array([4], [0, 255])` |
| `$urandom_range(0, 100)` | `randomize()` with constraints |

---
## 13. 연습문제

In [ ]:
# 연습 1: 패킷 생성기
# - packet_type: 0=DATA, 1=ACK, 2=NACK, 3=CTRL
# - payload_size: 0~1024, DATA일 때만 > 0
# - priority: 0~3

# TODO: @vsc.randobj 클래스 작성

# @vsc.randobj
# class packet:
#     def __init__(self):
#         pass
#     
#     @vsc.constraint
#     def packet_c(self):
#         pass

In [ ]:
# 연습 2: 패킷 coverage
# - packet_type별 bin
# - priority별 bin
# - packet_type x priority cross coverage

# TODO: @vsc.covergroup 클래스 작성

# @vsc.covergroup
# class packet_coverage:
#     def __init__(self):
#         pass

In [ ]:
vsc.rand_list_t(vsc.uint8_t(), 16)